In [ ]:
# run specs
# ---------
h11s = [5,6,7,8,9,10]

# for the polytopes sorted in order of increasing h21,
# study polys[start_ind:end_ind]
start_ind = -50
end_ind   = None

# Zp specs
dilations = [40]#[1,10,20,30,40]
num_pvecs = 200000
max_Kperp_gcd = 1

# Imports

CYTools

In [ ]:
from cytools import fetch_polytopes, Polytope
from cytools import Cone

Cornell-dev (making conifold objects)

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from projects.kklt.kklt_lib import kklt_conifolds

Local to PFV repo

In [ ]:
import sys; sys.path.append('..')
from pfvs import cydata, Zp, pfv

External

In [ ]:
import duckdb
import numpy as np
from tqdm.auto import tqdm

# Construct the conifolds

Read data from earlier KKLT runs (has info about which coni hosts an SG)

In [ ]:
con = duckdb.connect('kklt_db.duckdb')
tmp = con.execute("SHOW ALL TABLES").fetchdf()
df_vertices = con.execute("SELECT * FROM vertices").fetchdf()
df_conis    = con.execute("SELECT * FROM conifolds").fetchdf()
df_sgs      = con.execute("SELECT * FROM starting_guesses").fetchdf()
con.close()

Construct all polytopes with the desired h11

In [ ]:
all_polys = dict()

for h11 in h11s:
    tmp = []
    for _,row_df in df_conis[df_conis['hsmall']==h11].iterrows():
        mask  = ((df_vertices['hsmall']==row_df['hsmall']) & (df_vertices['hlarge']==row_df['hlarge']) & (df_vertices['poly_idx']==row_df['poly_idx']))
        verts = np.vstack(list(df_vertices[mask][f'coord{i}'].tolist() for i in range(4))).T
        poly  = Polytope(verts)
        tmp.append(poly)

    all_polys[h11] = tmp

Construct the conifolds, cut by whether they have SGs...

In [ ]:
if True:
    conis = dict()
    numLG = dict()
    numBG = dict()
    for h11 in h11s:
        conis_tmp = []
        numLG_tmp = []
        numBG_tmp = []

        for p in tqdm(all_polys[h11][start_ind:end_ind]):
            any_conis = False
            for coniclass in kklt_conifolds.kklt_conifolds(p.dual()):
                any_conis = True
    
                # check if the coni has SGs
                # -------------------------
                # get the coni index
                coni_mask = (df_conis['hsmall']==p.h11("N")) &\
                            (df_conis['hlarge']==p.h21("N")) &\
                            (df_conis['poly_idx']==p.ks_ind()[2]) &\
                            (df_conis['one_face_divisor']==coniclass.one_face_divisors()[0])
                df_tmp = df_conis[coni_mask]
    
                if len(df_tmp)!=1:
                    print('weird error...')
                    print(len(df_tmp))
                    continue
                coni_idx = df_tmp['coni_idx'].tolist()[0]
    
                # check if there are any SGs
                sg_mask =   (df_sgs['hsmall']==p.h11("N")) &\
                            (df_sgs['hlarge']==p.h21("N")) &\
                            (df_sgs['poly_idx']==p.ks_ind()[2]) &\
                            (df_sgs['coni_idx']==coni_idx)
                df_tmp = df_sgs[sg_mask]
                df_tmp = df_tmp[df_tmp['dead']==0]
    
                # save the conifold if it has SGs
                # -------------------------------
                if len(df_tmp):
                    for coni in coniclass.conifolds():
                        conis_tmp.append(coni)
                        numLG_tmp.append(sum(df_tmp['inLG']))
                        numBG_tmp.append(sum(df_tmp['inLG']==0))
        
            if any_conis==False:
                print('p bad :(')

        conis[h11] = conis_tmp
        numLG[h11] = numLG_tmp
        numBG[h11] = numBG_tmp

# Search in the conifolds

In [ ]:
import joblib

In [ ]:
import time

In [ ]:
from pfvs import coniZp
import importlib
importlib.reload(coniZp)

## Helpers

In [ ]:
def coni_to_ps(coni, min_N_pts=None):
    # prep the data for PFV runs
    # --------------------------
    cy   = coni.dual_triangulation().cy()
    data = cydata.CYData.from_cy(cy, coni_curve=coni.conifold_charge())

    # get p-vectors
    ps = Zp.pvecs(
        data,
        min_N_pts=num_pvecs if min_N_pts is None else min_N_pts,
        verbosity=0
    )

    return ps

In [ ]:
def ps_to_KMs(coni, ps, dilation):
    cy   = coni.dual_triangulation().cy()
    data = cydata.CYData.from_cy(cy, coni_curve=coni.conifold_charge())

    # construct PFVs
    pfvs = Zp.coniZpM(
        data=data,
        ps=ps,
        M0min=13,
        Qmax=data.h11+data.h21+4,
        ellipsoid_dilation=dilation,
        use_gcd_lattice=False,
        max_Kperp_gcd=max_Kperp_gcd,
        max_N_pfvs=1_000_000_000,
        return_formal_pfvs=True,
        verbosity=0)

    return pfvs

In [ ]:
import json

In [ ]:
  import json
                                                                                                                      
  def _fmt(obj, depth=0, pad='  '):
      ind  = pad * (depth + 1)                                                                                        
      cind = pad * depth
      if isinstance(obj, list) and all(isinstance(x, (int, float)) for x in obj):                                     
          return '[' + ', '.join(str(x) for x in obj) + ']'                                                           
      elif isinstance(obj, list):                                                                                     
          if not obj:                                                                                                 
              return '[]'
          items = [ind + _fmt(x, depth + 1, pad) for x in obj]                                                        
          return '[\n' + ',\n'.join(items) + '\n' + cind + ']'
      elif isinstance(obj, dict):                                                                                     
          if not obj:
              return '{}'                                                                                             
          items = [ind + json.dumps(k) + ': ' + _fmt(v, depth + 1, pad) for k, v in obj.items()]
          return '{\n' + ',\n'.join(items) + '\n' + cind + '}'                                                        
      else:
          return json.dumps(obj)                                                                                      
                  
  def compact_json(obj):                                                                                              
      if isinstance(obj, list):
          return '[\n' + ',\n'.join('  ' + _fmt(x, 1) for x in obj) + '\n]\n'                                         
      return _fmt(obj, 0) + '\n'                                                                                      
  
  def extract_example(coni, note=""):                                                                                 
      cy   = coni.dual_triangulation().cy()
      data = cydata.CYData.from_cy(cy, coni_curve=coni.conifold_charge())                                             
      return {                                                                                                        
          "note":       note,                                                                                         
          "h11":        int(data.h11),                                                                                
          "h21":        int(data.h21),
          "kappa":      data.kappa.tolist(),                                                                          
          "c2":         [int(x) for x in data.c2],
          "H":          data.H.tolist(),                                                                              
          "coni_curve": [int(x) for x in data.coni_curve],                                                            
          "cob":        data.cob.tolist(),
      }                                                                                                               
                  
  def already_exists(examples, new):                                                                                  
      for e in examples:
          if e["kappa"] == new["kappa"] and e["c2"] == new["c2"]:                                                     
              return True                                                                                             
      return False
                                                                                                                      
  examples_path = "../tests/examples.json"                                                                            
  with open(examples_path) as f:
      examples = json.load(f)                                                                                         
                  
  n_before = len(examples)                                                                                            
  for h11 in h11s:
      for coni in conis[h11]:                                                                                         
          entry = extract_example(coni)
          if not already_exists(examples, entry):                                                                     
              examples.append(entry)
                                                                                                                      
  with open(examples_path, "w") as f:
      f.write(compact_json(examples))
                                                                                                                      
  print(f"{len(examples) - n_before} new examples added ({len(examples)} total)")     

In [ ]:
len(conis[5])

In [ ]:
extract_example(conis[5][0])

## Make p-vectors

In [ ]:
time0_p = time.time() 
all_ps  = dict()
for h11 in tqdm(h11s):
    all_ps[h11] = joblib.Parallel(n_jobs=12)(joblib.delayed(coni_to_ps)(coni) for coni in conis[h11])
time1_p = time.time() 

In [ ]:
print(time1_p-time0_p)

Make the PFVs

In [ ]:
df_h11s       = []
df_dilations  = []
df_times      = []
df_pfvs       = []
df_num_pfvs   = []

for h11 in h11s:
    for coni,ps in tqdm(zip(conis[h11],all_ps[h11]), total=len(conis[h11])):
        for dilation in dilations:
            tic = time.time()
            pfvs = ps_to_KMs(coni, ps, dilation)
            toc = time.time()

            # save the data
            df_h11s.append(h11)
            df_dilations.append(dilation)
            df_times.append(toc-tic)
            df_pfvs.append(pfvs)
            df_num_pfvs.append(len(pfvs))

In [ ]:
pvecs_sorted = [sorted([pfv.pgrading.tolist() for pfv in pfvs_coni]) for pfvs_coni in df_pfvs]

In [ ]:
req_pvecs = []
for coni_i in range(len(conis[5])):
    coni = conis[5][coni_i]
    cy   = coni.dual_triangulation().cy()
    data = cydata.CYData.from_cy(cy, coni_curve=coni.conifold_charge())

    req_pvecs.append([])
    for p in pvecs_sorted[coni_i]:
        req_pvecs[-1].append(len(Zp.pvec_kernel(B=max(map(abs,p)),
                        linmat=data.H_cob.astype(np.int32),
                        linmin=1,
                        max_N_out=1000000,
                        max_N_iter=1000000)[0]))

In [ ]:
req_pvecs_plot

In [ ]:
pdenoms = np.rint(extract_p_denominators("foo.txt")).astype(int)
pdenom_odd = [pdenom for pdenom in pdenoms if pdenom%2==1]

In [ ]:
pdenom_odd

In [ ]:
req_pvecs_plot = [N for req_pvecs_coni in req_pvecs for N in req_pvecs_coni]
    
plt.hist(
    req_pvecs_plot,
    bins=np.arange(min(req_pvecs_plot)-0.5,max(req_pvecs_plot)+1.5, step=2000),
    histtype='step')
plt.yscale('log')
#plt.xscale('log')

plt.ylabel('# PFVs')
plt.xlabel('required number of p-vectors')

In [ ]:
req_pvecs = [n for req_coni in req_pvecs for n in req_coni]

In [ ]:
pfvs_concat = [pfv for pfvs_coni in df_pfvs for pfv in pfvs_coni]

In [ ]:
coni = conis[5][34]
cy   = coni.dual_triangulation().cy()
data = cydata.CYData.from_cy(cy, coni_curve=coni.conifold_charge())

In [ ]:
max(map(abs, pvecs_sorted[34][0]))

In [ ]:
Zp.pvec_kernel(B=5,
                    linmat=data.H_cob.astype(np.int32),
                    linmin=1,
                    max_N_out=1000000,
                    max_N_iter=1000000)

In [ ]:
req_pvecs = []
for p in pfvs_sorted[34]:
    req_pvecs.append(len(Zp.pvec_kernel(B=max(map(abs,p)),
                    linmat=data.H_cob.astype(np.int32),
                    linmin=1,
                    max_N_out=1000000,
                    max_N_iter=1000000)[0]))

In [ ]:
plt.hist(req_pvecs)
plt.yscale('log')

In [ ]:
np.argmax(list(map(len, pfvs_sorted)))

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'h11': df_h11s,
    'dilation': df_dilations,
    'time':df_times,
    'gcdlattice':df_gcdlattice,
    'numpfv':df_num_pfvs
})

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(df['h11'], df['time'], s=1)

In [ ]:
df_tmp = df[(df['gcdlattice']==1) & (df['dilation']==40)]
plt.scatter(df_tmp['h11'], df_tmp['time'], label='yesgcd')

df_tmp = df[(df['gcdlattice']==0) & (df['dilation']==40)]
plt.scatter(df_tmp['h11'], df_tmp['time'],label='nogcd')

plt.legend()

In [ ]:
time0_pfv = time.time()
KMs   = joblib.Parallel(n_jobs=12)(
    joblib.delayed(ps_to_KMs)(coni, ps)\
    for coni,ps in zip(conis,all_ps)
)
time1_pfv = time.time()

In [ ]:
time1_pfv-time0_pfv

# Saving stuff...

In [ ]:
def save_as_formal_pfvs(conis, all_KMs, numLG, numBG):
    """
    "A huge list of the form [verts, heights, conifold, cob, PFVs],
    where the number of lines in the list is the number of conifolds
    you searched."

    Format of PFVs is PFVs = [[M,K], [M,K], ...], with one [M,K] per
    PFV.
    """
    all_pfvs_concat = []
    numLG_concat    = []
    numBG_concat    = []
    for coni, KMs, nLG, nBG in zip(conis, all_KMs, numLG, numBG):
        cy   = coni.dual_triangulation().cy()
        data = cydata.CYData.from_cy(cy, coni_curve=coni.conifold_charge())

        # get verts, heights
        # ------------------
        verts   = data.vertices
        heights = data.heights
    
        # get coni data
        # -------------
        coninop = data.coni_curve
        cob     = data.cob

        # get PFV data
        # ------------
        Ks = np.array(KMs)[:,:data.h11]
        Ms = np.array(KMs)[:,data.h11:]

        for K,M in zip(Ks,Ms):
            pfv = diagnostics.PFV(data, K, M)
            all_pfvs_concat.append(pfv)
            numLG_concat.append(nLG)
            numBG_concat.append(nBG)

    # save it!
    # --------
    with open("PFVSTRINGS-" + fname + ".txt", 'w') as f:
        for pfv,nLG,nBG in zip(all_pfvs_concat,numLG_concat,numBG_concat):
            f.write(f"{pfv}\n(#LG,#BG) SGs = {(nLG, nBG)}\n\n")

    return all_pfvs_concat

In [ ]:
all_pfvs_concat = save_as_formal_pfvs(conis, KMs, numLG, numBG)

In [ ]:
for pfv in all_pfvs_concat:
    assert pfv.check_all()

In [ ]:
def save_for_richard(conis, all_KMs, numLG, numBG):
    """
    "A huge list of the form [verts, heights, conifold, cob, PFVs],
    where the number of lines in the list is the number of conifolds
    you searched."

    Format of PFVs is PFVs = [[M,K], [M,K], ...], with one [M,K] per
    PFV.
    """
    output = []
    for coni, KMs, nLG, nBG in zip(conis, all_KMs, numLG, numBG):
        cy   = coni.dual_triangulation().cy()
        data = cydata.CYData.from_cy(cy, coni_curve=coni.conifold_charge())

        # get verts, heights
        # ------------------
        verts   = data.vertices
        heights = data.heights
    
        # get coni data
        # -------------
        coninop = data.coni_curve
        cob     = data.cob

        # get PFV data
        # ------------
        Ks = np.array(KMs)[:,:data.h11]
        Ms = np.array(KMs)[:,data.h11:]
        
        pfvs = []
        for K,M in zip(Ks,Ms):
            pfv = diagnostics.PFV(data, K, M)
            pfvs.append([
                [int(Ki) for Ki in K],
                [int(Mi) for Mi in M],
                [float(pi) for pi in pfv.p]
            ])

        # save to output array
        # --------------------
        output.append([
            verts,
            heights,
            coninop.tolist(),
            cob.tolist(),
            nLG,
            nBG,
            pfvs
        ])

    # save
    with open("forRICHARD_PFVSTRINGS-" + fname + ".txt", 'w') as f:
        for conidata in output:
            f.write(str(conidata)+'\n')

In [ ]:
save_for_richard(conis, KMs, numLG, numBG)